# Function Calling Fine-Tuning: Gemma4-E4B on xLAM Dataset (Unsloth)

**Author:** Behrooz Azarkhalili

This notebook fine-tunes **Gemma4-E4B** (~4B params) for function calling using the
[Salesforce/xlam-function-calling-60k](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k) dataset
with **Unsloth** for 2x faster training and 60% less VRAM.

| Feature | Detail |
|---------|--------|
| Framework | Unsloth (FastModel) |
| Model | `unsloth/gemma-4-E4B-it` |
| Dataset | xLAM 60K function calling examples |
| Method | LoRA (4-bit QLoRA) |
| GGUF Export | Built-in via `save_pretrained_gguf()` |

### Hardware
- **Minimum:** 10 GB VRAM (QLoRA)
- **Recommended:** 16+ GB for batch_size > 1

In [ ]:
# ============================================================================
# Install (uncomment for Colab / bare-metal)
# ============================================================================
# !pip install unsloth datasets tqdm

# SLURM (Fir cluster):
#   module load gcc arrow python/3.11.5
#   source /scratch/$USER/venvs/hf_unsloth/bin/activate

In [ ]:
from __future__ import annotations

import json
import os
import torch

# ── Smoke test flag ──
SMOKE_TEST: bool = False  # Set False for full training

# ── Configuration ──
MODEL_NAME = "unsloth/gemma-4-E4B-it"
# MODEL_NAME = "unsloth/gemma-4-E2B-it"  # Smaller variant

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
HUB_MODEL_ID = "ermiaazarkhalili/Gemma4-E4B-Function-Calling-xLAM-Unsloth"
LORA_R = 16
LORA_ALPHA = 16
LEARNING_RATE = 2e-4
MAX_STEPS = 5 if SMOKE_TEST else 1000
BATCH_SIZE = 2
GRAD_ACCUM = 4

print(f"[OK] Model: {MODEL_NAME}")
print(f"[OK] SMOKE_TEST: {SMOKE_TEST}")
print(f"[OK] max_steps: {MAX_STEPS}")
print(f"[OK] PyTorch: {torch.__version__}")
print(f"[OK] GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# ============================================================================
# HF Authentication
# ============================================================================
try:
    from dotenv import load_dotenv; load_dotenv()
except ImportError: pass
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception: pass

from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("[OK] Authenticated with HF Hub")
else:
    print("[WARN] No HF_TOKEN found")

In [ ]:
# ============================================================================
# Load Model with Unsloth (FastModel for Gemma4 text-only)
# ============================================================================
# Reference: unslothai/notebooks/python_scripts/Gemma4_(E4B)-Text.py
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect
    load_in_4bit=LOAD_IN_4BIT,
    full_finetuning=False,
)
print(f"[OK] Model loaded: {MODEL_NAME}")

In [ ]:
# ============================================================================
# Apply LoRA Adapters (Unsloth handles Gemma4ClippableLinear internally)
# ============================================================================
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,     # Text-only training
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)
print(f"[OK] LoRA applied: r={LORA_R}, alpha={LORA_ALPHA}")

In [ ]:
# ============================================================================
# Set Gemma4 Chat Template
# ============================================================================
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-4",
)
print(f"[OK] Chat template: gemma-4")

In [ ]:
# ============================================================================
# Load & Process xLAM Dataset
# ============================================================================
from datasets import load_dataset

def process_xlam_sample(row, tokenizer):
    """Format xLAM sample with <query>, <tools>, <answers> tags."""
    query = f"<query>{row['query']}</query>\n\n"
    try:
        tools = json.loads(row["tools"])
        tools_text = "\n".join(str(t) for t in tools)
    except Exception:
        tools_text = str(row["tools"])
    formatted_tools = f"<tools>{tools_text}</tools>\n\n"
    try:
        answers = json.loads(row["answers"])
        answers_text = "\n".join(str(a) for a in answers)
    except Exception:
        answers_text = str(row["answers"])
    formatted_answers = f"<answers>{answers_text}</answers>"
    text = query + formatted_tools + formatted_answers + tokenizer.eos_token
    return {"text": text}

print("[OK] Loading xLAM dataset ...")
dataset = load_dataset("Salesforce/xlam-function-calling-60k", split="train")
print(f"[OK] Loaded {len(dataset):,} samples")

if SMOKE_TEST:
    dataset = dataset.select(range(100))
    print(f"[SMOKE] Truncated to {len(dataset)} samples")

dataset = dataset.map(lambda row: process_xlam_sample(row, tokenizer), num_proc=4)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])
print(f"[OK] Processed: {len(dataset):,} samples with 'text' column")

# Preview
print(f"\n--- Sample (first 300 chars) ---")
print(dataset[0]["text"][:300])

In [ ]:
# ============================================================================
# Training with SFTTrainer
# ============================================================================
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

# GPU stats before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"[OK] GPU: {gpu_stats.name}, Max memory: {max_memory} GB")
print(f"[OK] Reserved before training: {start_gpu_memory} GB")

# Train
print(f"\n[....] Training ({MAX_STEPS} steps) ...")
trainer_stats = trainer.train()

# Report
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\n[OK] Training complete!")
print(f"  Runtime: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"  Loss:    {trainer_stats.metrics.get('train_loss', 'N/A')}")
print(f"  Peak VRAM: {used_memory} GB ({round(used_memory/max_memory*100, 1)}%)")

In [ ]:
# ============================================================================
# Inference Test
# ============================================================================
from transformers import TextStreamer

test_prompts = [
    "Check if the numbers 8 and 1233 are powers of two.",
    "What's the weather like in New York today?",
    "Calculate the average of these numbers: 10, 20, 30, 40, 50",
]

print("[....] Testing function calling ...\n")
for i, prompt in enumerate(test_prompts, 1):
    print(f"{'=' * 60}")
    print(f"Test {i}: {prompt}")
    print(f"{'=' * 60}")
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", tokenize=True, return_dict=True,
    ).to("cuda")
    outputs = model.generate(
        **inputs, max_new_tokens=256,
        temperature=1.0, top_p=0.95, top_k=64, use_cache=True,
    )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(response[:300])
    print()

In [ ]:
# ============================================================================
# Save & Export
# ============================================================================
import gc

# Save LoRA adapter
model.save_pretrained("gemma4_e4b_xlam_lora")
tokenizer.save_pretrained("gemma4_e4b_xlam_lora")
print("[OK] LoRA adapter saved")

# Merge and save full model
if not SMOKE_TEST:
    model.save_pretrained_merged(
        "gemma4_e4b_xlam_merged", tokenizer,
    )
    print("[OK] Merged model saved")

    # Push to Hub
    model.push_to_hub_merged(
        HUB_MODEL_ID, tokenizer,
        token=os.environ.get("HF_TOKEN", ""),
    )
    print(f"[OK] Pushed merged model to {HUB_MODEL_ID}")

# GGUF export
if not SMOKE_TEST:
    model.save_pretrained_gguf(
        "gemma4_e4b_xlam_gguf", tokenizer,
        quantization_method=["q4_k_m", "q8_0"],
    )
    print("[OK] GGUF saved (Q4_K_M + Q8_0)")

    model.push_to_hub_gguf(
        f"{HUB_MODEL_ID}-GGUF", tokenizer,
        quantization_method=["q4_k_m", "q8_0"],
        token=os.environ.get("HF_TOKEN", ""),
    )
    print(f"[OK] GGUF pushed to {HUB_MODEL_ID}-GGUF")
else:
    print("[SMOKE] Skipping merge/GGUF/push (SMOKE_TEST=True)")

print("\n[OK] Done!")